In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('divar.csv', low_memory=False)
df_sale = df[df['price_value'].notna()].copy()

def build_train_val_test(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
    X_train = X_train.copy()
    X_val = X_val.copy()
    X_test = X_test.copy()
    return X_train, X_val, X_test, y_train, y_val, y_test

def clean_pipeline(X_train, X_val, X_test):
    drop_cols = ['description', 'title', 'id', 'token']
    X_train = X_train.drop(columns=drop_cols, errors='ignore')
    X_val = X_val.drop(columns=drop_cols, errors='ignore')
    X_test = X_test.drop(columns=drop_cols, errors='ignore')

    missing_ratio = X_train.isna().mean()
    cols_to_drop = missing_ratio[missing_ratio > 0.6].index
    X_train = X_train.drop(columns=cols_to_drop)
    X_val = X_val.drop(columns=cols_to_drop, errors='ignore')
    X_test = X_test.drop(columns=cols_to_drop, errors='ignore')

    numeric_cols = X_train.select_dtypes(include='number').columns
    categorical_cols = X_train.select_dtypes(exclude='number').columns

    for col in numeric_cols:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_val[col] = X_val[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)

    for col in categorical_cols:
        mode_val = X_train[col].mode()[0]
        X_train[col] = X_train[col].fillna(mode_val).infer_objects(copy=False)
        X_val[col] = X_val[col].fillna(mode_val).infer_objects(copy=False)
        X_test[col] = X_test[col].fillna(mode_val).infer_objects(copy=False)


    return X_train, X_val, X_test

def get_percentile_bounds(series, lower_pct=0.01, upper_pct=0.99):
    lower = series.quantile(lower_pct)
    upper = series.quantile(upper_pct)
    return lower, upper

def outlier_pipeline(X_train, X_val, X_test, col):
    lower, upper = get_percentile_bounds(X_train[col])
    X_train[col] = X_train[col].clip(lower, upper)
    X_val[col] = X_val[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)
    return X_train, X_val, X_test

def encode_pipeline(X_train, X_val, X_test):
    high_card_cols = ['city_slug', 'neighborhood_slug']
    for col in high_card_cols:
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col].astype(str))
        X_val[col] = X_val[col].astype(str).map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
        X_test[col] = X_test[col].astype(str).map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)

    low_card_cols = ['floor_material', 'deed_type', 'property_type', 'building_direction']
    low_card_cols = [col for col in low_card_cols if col in X_train.columns]

    X_train = pd.get_dummies(X_train, columns=low_card_cols)
    X_val = pd.get_dummies(X_val, columns=low_card_cols)
    X_test = pd.get_dummies(X_test, columns=low_card_cols)

    X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    return X_train, X_val, X_test


X_train_sale, X_val_sale, X_test_sale, y_train_sale, y_val_sale, y_test_sale = build_train_val_test(df_sale, 'price_value')
X_train_sale, X_val_sale, X_test_sale = clean_pipeline(X_train_sale, X_val_sale, X_test_sale)
X_train_sale, X_val_sale, X_test_sale = outlier_pipeline(X_train_sale, X_val_sale, X_test_sale, 'building_size')
X_train_sale, X_val_sale, X_test_sale = encode_pipeline(X_train_sale, X_val_sale, X_test_sale)


/var/folders/zh/mb100nyn3rs7s19dxvg2x_kh0000gn/T/ipykernel_9776/3530919425.py:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train[col] = X_train[col].fillna(mode_val).infer_objects(copy=False)
/var/folders/zh/mb100nyn3rs7s19dxvg2x_kh0000gn/T/ipykernel_9776/3530919425.py:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_val[col] = X_val[col].fillna(mode_val).infer_objects(copy=False)
/var/folders/zh/mb100nyn3rs7s19dxvg2x_kh0000gn/T/ipykernel_9776/3530919425.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is depreca